In [ ]:
# In[1]:

import pandas as pd
import pytz

# timezone for UTC+8
tz = pytz.timezone('Asia/Shanghai')

# load files (they are in current working directory)
df_metric = pd.read_csv("metric.csv")
df_trace = pd.read_csv("trace.csv")
df_log = pd.read_csv("log.csv")
df_error = pd.read_csv("error_logs.csv")

# helper to build summary for a generic dataframe
def summarize_df(df, file_label, cmdb_col='cmdb_id', name_col=None, name_label='name'):
    total_rows = len(df)
    unique_cmdb = df[cmdb_col].nunique() if cmdb_col in df.columns else 0
    cmdb_samples = sorted(df[cmdb_col].dropna().unique())[:20] if cmdb_col in df.columns else []
    unique_names = df[name_col].nunique() if (name_col and name_col in df.columns) else 0
    name_samples = sorted(df[name_col].dropna().unique())[:20] if (name_col and name_col in df.columns) else []
    # timestamps
    if 'timestamp' in df.columns and len(df)>0:
        min_ts = int(df['timestamp'].min())
        max_ts = int(df['timestamp'].max())
        # convert epoch (assumed UTC) to Asia/Shanghai tz-aware datetime strings
        min_ts_local = pd.to_datetime(min_ts, unit='s', utc=True).tz_convert(tz)
        max_ts_local = pd.to_datetime(max_ts, unit='s', utc=True).tz_convert(tz)
        min_ts_str = min_ts_local.isoformat()
        max_ts_str = max_ts_local.isoformat()
    else:
        min_ts_str = None
        max_ts_str = None
    summary = pd.DataFrame([{
        'file': file_label,
        'total_rows': total_rows,
        'unique_cmdb_count': unique_cmdb,
        'cmdb_samples(first<=20)': cmdb_samples,
        f'unique_{name_label}_count': unique_names,
        f'{name_label}_samples(first<=20)': name_samples,
        'min_timestamp(Asia/Shanghai)': min_ts_str,
        'max_timestamp(Asia/Shanghai)': max_ts_str
    }])
    top5 = df.head(5)
    return summary, top5

# create summaries for each file
metric_summary, metric_top5 = summarize_df(df_metric, 'metric.csv', cmdb_col='cmdb_id', name_col='kpi_name', name_label='kpi')
trace_summary, trace_top5 = summarize_df(df_trace, 'trace.csv', cmdb_col='cmdb_id', name_col='trace_name', name_label='trace')
log_summary, log_top5 = summarize_df(df_log, 'log.csv', cmdb_col='cmdb_id', name_col='log_name', name_label='log')
error_summary, error_top5 = summarize_df(df_error, 'error_logs.csv', cmdb_col='cmdb_id', name_col='message', name_label='message')

# Display compact summaries and top5 rows for each file
metric_summary, metric_top5, trace_summary, trace_top5, log_summary, log_top5, error_summary, error_top5

```
Out[1]:
```
summary = (
    "metric.csv: 319,703 rows; 59 unique cmdb_id (samples: adservice, adservice-0, adservice-1, adservice-2, adservice-... up to 20); "
    "1,368 unique kpi_name (samples: app.grpc.count, app.grpc.mrt, app.grpc.rr, app.grpc.sr, app.http.count, ... up to 20); "
    "min timestamp 2022-03-20T16:00:00+08:00; max timestamp 2022-03-20T17:00:00+08:00. Top 5 rows were displayed.\n\n"
    "trace.csv: 31,476 rows; 40 unique cmdb_id (samples: adservice-0, adservice-1, adservice-2, adservice-... up to 20); "
    "200 unique trace_name (samples include trace.from_checkoutservice-0.duration_mean, trace.from_frontend-0.duration_mean, ... up to 20); "
    "min timestamp 2022-03-20T16:30:00+08:00; max timestamp 2022-03-20T16:59:00+08:00. Top 5 rows were displayed.\n\n"
    "log.csv: 1,772 rows; 31 unique cmdb_id (samples: adservice-0, adservice-1, adservice-2, cartservice-... up to 20); "
    "2 unique log_name (log.error_count, log.row_count); "
    "min timestamp 2022-03-20T16:30:00+08:00; max timestamp 2022-03-20T17:00:00+08:00. Top 5 rows were displayed.\n\n"
    "error_logs.csv: 60 rows; 3 unique cmdb_id (frontend-0, frontend-1, frontend-2); "
    "1 unique message sample (\"severity: error, message: request error\"); "
    "min timestamp 2022-03-20T16:32:50+08:00; max timestamp 2022-03-20T16:37:13+08:00. Top 5 rows were displayed."
)
summary

The original code execution output of IPython Kernel is also provided below for reference:

(         file  total_rows  unique_cmdb_count                            cmdb_samples(first<=20)  unique_kpi_count                             kpi_samples(first<=20) min_timestamp(Asia/Shanghai) max_timestamp(Asia/Shanghai)
0  metric.csv      319703                 59  [adservice, adservice-0, adservice-1, adservic...              1368  [app.grpc.count, app.grpc.mrt, app.grpc.rr, ap...    2022-03-20T16:00:00+08:00    2022-03-20T17:00:00+08:00,     timestamp    cmdb_id        kpi_name       value
0  1647763200  adservice  app.grpc.count  233.000000
1  1647763200  adservice    app.grpc.mrt    0.002522
2  1647763200  adservice     app.grpc.rr  100.000000
3  1647763200  adservice     app.grpc.sr  100.000000
4  1647763200  adservice  app.http.count   24.000000,         file  total_rows  unique_cmdb_count                            cmdb_samples(first<=20)  unique_trace_count                           trace_samples(first<=20) min_timestamp(Asia/Shanghai) max_timestamp(Asia/Shanghai)
0  trace.csv       31476                 40  [adservice-0, adservice-1, adservice-2, adserv...                 200  [trace.from_checkoutservice-0.duration_mean, t...    2022-03-20T16:30:00+08:00    2022-03-20T16:59:00+08:00,     timestamp      cmdb_id                           trace_name      value
0  1647765000  adservice-0  trace.from_frontend-0.duration_mean   0.000017
1  1647765000  adservice-0   trace.from_frontend-0.duration_p95   0.000025
2  1647765000  adservice-0     trace.from_frontend-0.error_rate   0.000000
3  1647765000  adservice-0      trace.from_frontend-0.row_count  29.000000
4  1647765000  adservice-0  trace.from_frontend-1.duration_mean   0.000020,       file  total_rows  unique_cmdb_count                            cmdb_samples(first<=20)  unique_log_count            log_samples(first<=20) min_timestamp(Asia/Shanghai) max_timestamp(Asia/Shanghai)
0  log.csv        1772                 31  [adservice-0, adservice-1, adservice-2, cartse...                 2  [log.error_count, log.row_count]    2022-03-20T16:30:00+08:00    2022-03-20T17:00:00+08:00,     timestamp      cmdb_id         log_name  value
0  1647765000  adservice-0  log.error_count    0.0
1  1647765000  adservice-0    log.row_count  142.0
2  1647765000  adservice-1  log.error_count    0.0
3  1647765000  adservice-1    log.row_count  132.0
4  1647765000  adservice-2  log.error_count    0.0,              file  total_rows  unique_cmdb_count               cmdb_samples(first<=20)  unique_message_count                 message_samples(first<=20) min_timestamp(Asia/Shanghai) max_timestamp(Asia/Shanghai)
0  error_logs.csv          60                  3  [frontend-0, frontend-1, frontend-2]                     1  [severity: error, message: request error]    2022-03-20T16:32:50+08:00    2022-03-20T16:37:13+08:00,     timestamp     cmdb_id                                  message
0  1647765170  frontend-0  severity: error, message: request error
1  1647765170  frontend-1  severity: error, message: request error
2  1647765172  frontend-0  severity: error, message: request error
3  1647765174  frontend-0  severity: error, message: request error
4  1647765174  frontend-0  severity: error, message: request error)```
```

In [ ]:
# In[2]:

import pandas as pd
import numpy as np
import pytz

# reuse df_metric if exists, otherwise load
try:
    df_metric
except NameError:
    df_metric = pd.read_csv("metric.csv")

# ensure correct types
df_metric['value'] = pd.to_numeric(df_metric['value'], errors='coerce')
df_metric = df_metric.dropna(subset=['value'])

# timezone
tz = pytz.timezone('Asia/Shanghai')

# 1) Global P05 and P95 per (cmdb_id, kpi_name) using entire metric.csv
thresholds = df_metric.groupby(['cmdb_id', 'kpi_name'])['value'].agg(
    p05=lambda x: x.quantile(0.05),
    p95=lambda x: x.quantile(0.95)
).reset_index().set_index(['cmdb_id', 'kpi_name'])

# 2) Filter metric.csv to incident window 2022-03-20 16:30:00 to 2022-03-20 17:00:00 (Asia/Shanghai)
start_local = tz.localize(pd.to_datetime("2022-03-20 16:30:00"))
end_local = tz.localize(pd.to_datetime("2022-03-20 17:00:00"))
# convert to UTC epoch seconds since stored timestamps are epoch UTC
start_epoch = int(start_local.astimezone(pytz.UTC).timestamp())
end_epoch = int(end_local.astimezone(pytz.UTC).timestamp())

df_window = df_metric[(df_metric['timestamp'] >= start_epoch) & (df_metric['timestamp'] <= end_epoch)].copy()

# 3) Merge thresholds and detect anomalies where value > p95 OR value < p05
df_join = df_window.merge(thresholds, left_on=['cmdb_id','kpi_name'], right_index=True, how='left')

# safe ratio computation: only when p95>0
df_join['is_anom'] = (df_join['value'] > df_join['p95']) | (df_join['value'] < df_join['p05'])
df_join['ratio_over_p95'] = np.where(
    (df_join['p95'] > 0) & (df_join['value'] > df_join['p95']),
    df_join['value'] / df_join['p95'],
    np.nan
)

# 4) Aggregate per (cmdb_id, kpi_name)
agg = df_join.groupby(['cmdb_id','kpi_name']).apply(
    lambda g: pd.Series({
        'anomaly_count': int(g['is_anom'].sum()),
        'earliest_anom_epoch': int(g.loc[g['is_anom'],'timestamp'].min()) if g['is_anom'].any() else np.nan,
        'earliest_anom_asia_shanghai': pd.to_datetime(int(g.loc[g['is_anom'],'timestamp'].min()), unit='s', utc=True).tz_convert(tz).isoformat() if g['is_anom'].any() else None,
        'max_in_window': float(g['value'].max()),
        'min_in_window': float(g['value'].min()),
        'max_ratio_over_p95': float(g['ratio_over_p95'].max(skipna=True)) if g['ratio_over_p95'].notna().any() else 0.0
    })
).reset_index()

# keep only those with at least one anomaly
agg_anom = agg[agg['anomaly_count'] > 0].copy()

# 5) Top 20 rows sorted by anomaly_count desc
agg_top20 = agg_anom.sort_values(by='anomaly_count', ascending=False).head(20).reset_index(drop=True)

# Summary counts and unique cmdb_id among anomalous pairs (limit to first 20)
total_anomalous_pairs = agg_anom.shape[0]
unique_cmdbs_with_anom = sorted(agg_anom['cmdb_id'].unique())[:20]

# compact outputs
agg_top20_selected = agg_top20[['cmdb_id','kpi_name','anomaly_count','earliest_anom_epoch','earliest_anom_asia_shanghai','max_in_window','min_in_window','max_ratio_over_p95']]

agg_top20_selected, total_anomalous_pairs, unique_cmdbs_with_anom

```
Out[2]:
```
```python
summary = (
    "Analysis (metric.csv) using global P05/P95 thresholds and incident window 2022-03-20 16:30:00 to 17:00:00 (Asia/Shanghai):\n"
    "- Total distinct (cmdb_id, kpi_name) with ≥1 anomaly: 1705.\n"
    "- Top anomalous pairs all had 6 anomaly points each (examples):\n"
    "  * frontend-1 | mesh.source.frontend.cartservice... — earliest 2022-03-20T16:33:00+08:00; max 89.5; min 4.0; max ratio over P95 ≈ 1.065\n"
    "  * checkoutservice | app.grpc.count — earliest 2022-03-20T16:33:00+08:00; max 21.0; min 2.0; ratio ≈ 1.105\n"
    "  * recommendationservice | app.grpc.count — earliest 2022-03-20T16:33:00+08:00; max 324.0; min 94.0; ratio ≈ 1.025\n"
    "  * frontend-2 | mesh.destination.unknown.frontend.istio_request... — earliest 2022-03-20T16:32:00+08:00; max 354.007; max ratio ≈ 10.30\n"
    "  * checkoutservice-1 | mesh.destination.frontend.checkoutservice... — earliest 2022-03-20T16:31:00+08:00; max ratio ≈ 204.996 (very large spike)\n"
    "- Many top anomalies cluster around 16:31–16:33+08:00 (earliest anomaly times shown above).\n"
    "- Affected component examples (unique cmdb_id among anomalous pairs, first 20): "
    "['adservice', 'adservice-0', 'adservice-1', 'adservice-2', 'adservice2', 'adservice2-0', 'cartservice', 'cartservice-0', 'cartservice-1', 'cartservice-2', 'cartservice2-0', 'checkoutservice', 'checkoutservice-0', 'checkoutservice-1', 'checkoutservice-2', 'checkoutservice2-0', 'currencyservice', 'currencyservice-0', 'currencyservice-1', 'currencyservice-2'].\n"
    "- Notes: anomalies are widespread across mesh/request KPIs, services, pods, and some node-level metrics (e.g., node-1, node-5). A few entries show very large ratio_over_P95 (indicative of strong spikes) and warrant immediate inspection (examples: checkoutservice-1, frontend-2)."
)
summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(                  cmdb_id                                           kpi_name  anomaly_count  earliest_anom_epoch earliest_anom_asia_shanghai  max_in_window  min_in_window  max_ratio_over_p95
0              frontend-1  mesh.source.frontend.cartservice.istio_respons...            6.0         1.647765e+09   2022-03-20T16:33:00+08:00      89.500000       4.000000            1.065476
1         checkoutservice                                     app.grpc.count            6.0         1.647765e+09   2022-03-20T16:33:00+08:00      21.000000       2.000000            1.105263
2   recommendationservice                                     app.grpc.count            6.0         1.647765e+09   2022-03-20T16:33:00+08:00     324.000000      94.000000            1.025316
3              frontend-1  mesh.source.frontend.productcatalogservice.ist...            6.0         1.647765e+09   2022-03-20T16:33:00+08:00     471.000000      17.500000            1.057239
4              frontend-1  mesh.source.frontend.cartservice.istio_request...            6.0         1.647765e+09   2022-03-20T16:33:00+08:00       0.179375       0.007175            1.041969
5              frontend-1  mesh.source.frontend.currencyservice.istio_req...            6.0         1.647765e+09   2022-03-20T16:33:00+08:00     309.500000       5.000000            1.091711
6              frontend-1  mesh.source.frontend.recommendationservice.ist...            6.0         1.647765e+09   2022-03-20T16:33:00+08:00   20075.000000     687.500000            1.050360
7              frontend-2  mesh.destination.unknown.frontend.istio_reques...            6.0         1.647765e+09   2022-03-20T16:32:00+08:00     354.006750       0.167000           10.301600
8              frontend-1  mesh.source.frontend.adservice.istio_request_b...            6.0         1.647765e+09   2022-03-20T16:33:00+08:00   90450.000000    2325.000000            1.031063
9              frontend-1  mesh.source.frontend.adservice.istio_request_d...            6.0         1.647765e+09   2022-03-20T16:33:00+08:00       0.201175       0.004575            1.053824
10      checkoutservice-1  mesh.destination.frontend.checkoutservice.isti...            6.0         1.647765e+09   2022-03-20T16:31:00+08:00      59.500000       0.000000          204.995693
11                 node-5                       system.net.packets_out.count            6.0         1.647765e+09   2022-03-20T16:31:00+08:00     195.580000      93.090000            1.024730
12                 node-5                        system.net.packets_in.count            6.0         1.647765e+09   2022-03-20T16:31:00+08:00     196.120000      97.140000            1.020236
13     currencyservice2-0         container.node-6.container_memory_usage_MB            6.0         1.647766e+09   2022-03-20T16:40:00+08:00      52.636719      51.292969            1.013234
14     currencyservice2-0   container.node-6.container_memory_working_set_MB            6.0         1.647766e+09   2022-03-20T16:40:00+08:00      46.523438      45.179688            1.014999
15             frontend-1  mesh.source.frontend.cartservice.istio_request...            6.0         1.647765e+09   2022-03-20T16:33:00+08:00  135625.000000    5425.000000            1.047904
16                 node-1                               system.cpu.pct_usage            6.0         1.647765e+09   2022-03-20T16:30:00+08:00      36.070000      11.980000            1.199534
17             frontend-1  mesh.source.frontend.recommendationservice.ist...            6.0         1.647765e+09   2022-03-20T16:33:00+08:00  113150.000000    3875.000000            1.050360
18             frontend-1  mesh.source.frontend.recommendationservice.ist...            6.0         1.647765e+09   2022-03-20T16:33:00+08:00       0.308150       0.010125            1.041223
19  productcatalogservice                                     app.grpc.count            6.0         1.647765e+09   2022-03-20T16:33:00+08:00    2449.000000     745.000000            1.023829, 1705, ['adservice', 'adservice-0', 'adservice-1', 'adservice-2', 'adservice2', 'adservice2-0', 'cartservice', 'cartservice-0', 'cartservice-1', 'cartservice-2', 'cartservice2-0', 'checkoutservice', 'checkoutservice-0', 'checkoutservice-1', 'checkoutservice-2', 'checkoutservice2-0', 'currencyservice', 'currencyservice-0', 'currencyservice-1', 'currencyservice-2'])```
```

In [ ]:
# In[3]:

import numpy as np
import pandas as pd
import pytz

# reuse df_trace and tz if available; otherwise load
try:
    df_trace
except NameError:
    df_trace = pd.read_csv("trace.csv")
try:
    tz
except NameError:
    tz = pytz.timezone('Asia/Shanghai')

# ensure numeric values and drop NA values
df_trace['value'] = pd.to_numeric(df_trace['value'], errors='coerce')
df_trace = df_trace.dropna(subset=['value'])

# 1) Global P05 and P95 per trace_name using the entire trace.csv
trace_thresholds = df_trace.groupby('trace_name')['value'].agg(
    p05=lambda x: x.quantile(0.05),
    p95=lambda x: x.quantile(0.95)
).reset_index().set_index('trace_name')

# 2) Filter to incident window 2022-03-20 16:30:00 - 2022-03-20 17:00:00 Asia/Shanghai
start_local = tz.localize(pd.to_datetime("2022-03-20 16:30:00"))
end_local = tz.localize(pd.to_datetime("2022-03-20 17:00:00"))
start_epoch = int(start_local.astimezone(pytz.UTC).timestamp())
end_epoch = int(end_local.astimezone(pytz.UTC).timestamp())

df_trace_window = df_trace[(df_trace['timestamp'] >= start_epoch) & (df_trace['timestamp'] <= end_epoch)].copy()

# 3) Merge thresholds and mark anomalies where value > p95 OR value < p05
dft = df_trace_window.merge(trace_thresholds, left_on='trace_name', right_index=True, how='left')

dft['is_anom'] = (dft['value'] > dft['p95']) | (dft['value'] < dft['p05'])
dft['ratio_over_p95'] = np.where(
    (dft['p95'] > 0) & (dft['value'] > dft['p95']),
    dft['value'] / dft['p95'],
    np.nan
)

# 4) Aggregate per (cmdb_id, trace_name)
agg_trace = dft.groupby(['cmdb_id','trace_name']).apply(
    lambda g: pd.Series({
        'anomaly_count': int(g['is_anom'].sum()),
        'earliest_anom_epoch': int(g.loc[g['is_anom'],'timestamp'].min()) if g['is_anom'].any() else np.nan,
        'earliest_anom_asia_shanghai': pd.to_datetime(int(g.loc[g['is_anom'],'timestamp'].min()), unit='s', utc=True).tz_convert(tz).isoformat() if g['is_anom'].any() else None,
        'max_in_window': float(g['value'].max()),
        'min_in_window': float(g['value'].min()),
        'max_ratio_over_p95': float(g['ratio_over_p95'].max(skipna=True)) if g['ratio_over_p95'].notna().any() else 0.0
    })
).reset_index()

# keep only those with at least one anomaly
agg_trace_anom = agg_trace[agg_trace['anomaly_count'] > 0].copy()

# 5) Top 20 rows sorted by anomaly_count desc
agg_trace_top20 = agg_trace_anom.sort_values(by='anomaly_count', ascending=False).head(20).reset_index(drop=True)
agg_trace_top20_selected = agg_trace_top20[['cmdb_id','trace_name','anomaly_count','earliest_anom_epoch','earliest_anom_asia_shanghai','max_in_window','min_in_window','max_ratio_over_p95']]

# summary totals and unique cmdb_id list (limit to first 20)
total_anomalous_pairs_trace = agg_trace_anom.shape[0]
unique_cmdbs_with_trace_anom = sorted(agg_trace_anom['cmdb_id'].unique())[:20]

# compact outputs
agg_trace_top20_selected, total_anomalous_pairs_trace, unique_cmdbs_with_trace_anom

```
Out[3]:
```
Summary of trace.csv analysis (global P05/P95; window 2022-03-20 16:30–17:00 Asia/Shanghai):

- Total distinct (cmdb_id, trace_name) with ≥1 anomaly in the window: 479.
- Top anomalous pairs (by anomaly_count), examples from the top 20:
  - frontend2-0 | trace.self.row_count — anomaly_count 24; earliest 2022-03-20T16:36:00+08:00; max 2910; min 688; max ratio_over_p95 ≈ 1.199
  - checkoutservice-1 | trace.from_frontend-1.duration_mean — anomaly_count 12; earliest 2022-03-20T16:35:00+08:00; max 1.511s; min 0.0326s; max ratio_over_p95 ≈ 32.378 (large spike)
  - adservice-2 | trace.from_frontend-1.duration_mean — anomaly_count 12; earliest 2022-03-20T16:30:00+08:00; max 0.000020; min 0.000013; ratio_over_p95 0 (indicates below-P05 anomalies)
  - frontend2-0 | trace.from_root.row_count — anomaly_count 12; earliest 2022-03-20T16:37:00+08:00; max 289; min 68; ratio ≈ 1.076
  - checkoutservice-1 | trace.from_frontend-1.duration_p95 — anomaly_count 12; earliest 2022-03-20T16:35:00+08:00; max ratio ≈ 32.120
  - productcatalogservice2-0 | trace.from_frontend2-0.row_count — anomaly_count 11; earliest 2022-03-20T16:40:00+08:00; max 1340; min 355; ratio ≈ 1.049
  - Many other top entries are checkoutservice / checkoutservice2 / cartservice traces and various frontend→service traces.

- Timing: earliest anomalies among top items cluster between ~16:30 and 16:40+08:00 (most between 16:30–16:38).

- Nature of anomalies:
  - A mix of high-side spikes (value > global P95) and low-side drops (value < global P05). 
  - Some traces show very large ratio_over_p95 (e.g., checkoutservice-1 duration metrics with ratio >30), indicating strong latency spikes. 
  - Other anomalies are low-side (ratio_over_p95 = 0), e.g., unusually low durations or counts.

- Example affected components (first 20 unique cmdb_id among anomalous trace pairs): 
  ['adservice-1', 'adservice-2', 'adservice2-0', 'cartservice-0', 'cartservice-1', 'cartservice-2', 'cartservice2-0', 'checkoutservice-0', 'checkoutservice-1', 'checkoutservice-2', 'checkoutservice2-0', 'currencyservice-0', 'currencyservice-1', 'currencyservice-2', 'currencyservice2-0', 'emailservice2-0', 'frontend-0', 'frontend-1', 'frontend-2', 'frontend2-0'].

Recommendation: prioritize investigating the high-ratio duration spikes (e.g., checkoutservice-1) and large row_count spikes (frontend2-0 / productcatalogservice2-0), and review both high-side and low-side anomalies for root-cause clues.

The original code execution output of IPython Kernel is also provided below for reference:

(                     cmdb_id                                  trace_name  anomaly_count  earliest_anom_epoch earliest_anom_asia_shanghai  max_in_window  min_in_window  max_ratio_over_p95
0                frontend2-0                        trace.self.row_count           24.0         1.647765e+09   2022-03-20T16:36:00+08:00    2910.000000     688.000000            1.198789
1          checkoutservice-1         trace.from_frontend-1.duration_mean           12.0         1.647765e+09   2022-03-20T16:35:00+08:00       1.511167       0.032561           32.377895
2                adservice-2         trace.from_frontend-1.duration_mean           12.0         1.647765e+09   2022-03-20T16:30:00+08:00       0.000020       0.000013            0.000000
3                frontend2-0                   trace.from_root.row_count           12.0         1.647765e+09   2022-03-20T16:37:00+08:00     289.000000      68.000000            1.075549
4          checkoutservice-1          trace.from_frontend-1.duration_p95           12.0         1.647765e+09   2022-03-20T16:35:00+08:00       1.511167       0.032561           32.119550
5              cartservice-2          trace.from_frontend-1.duration_p95           12.0         1.647765e+09   2022-03-20T16:34:00+08:00       0.001000       0.000000            0.000000
6               adservice2-0         trace.from_frontend2-0.duration_p95           11.0         1.647765e+09   2022-03-20T16:31:00+08:00       0.000033       0.000024            0.000000
7          checkoutservice-2          trace.from_frontend-1.duration_p95           11.0         1.647765e+09   2022-03-20T16:36:00+08:00       0.105557       0.034604            2.243593
8          checkoutservice-2          trace.from_frontend-2.duration_p95           11.0         1.647765e+09   2022-03-20T16:30:00+08:00       0.055658       0.040122            1.263834
9         checkoutservice2-0        trace.from_frontend2-0.duration_mean           11.0         1.647765e+09   2022-03-20T16:31:00+08:00       0.070878       0.010312            1.284634
10  productcatalogservice2-0            trace.from_frontend2-0.row_count           11.0         1.647766e+09   2022-03-20T16:40:00+08:00    1340.000000     355.000000            1.049375
11        checkoutservice2-0         trace.from_frontend2-0.duration_p95           11.0         1.647765e+09   2022-03-20T16:31:00+08:00       0.110452       0.010312            1.189120
12        checkoutservice2-0            trace.from_frontend2-0.row_count           11.0         1.647765e+09   2022-03-20T16:30:00+08:00      14.000000       1.000000            0.000000
13         checkoutservice-2         trace.from_frontend-2.duration_mean           11.0         1.647765e+09   2022-03-20T16:30:00+08:00       0.055658       0.040122            1.263834
14         checkoutservice-2         trace.from_frontend-1.duration_mean           10.0         1.647765e+09   2022-03-20T16:36:00+08:00       0.105557       0.034604            2.261638
15        checkoutservice2-0               trace.from_root.duration_mean           10.0         1.647765e+09   2022-03-20T16:31:00+08:00       0.004693       0.002642            0.000000
16                frontend-1  trace.to_productcatalogservice-1.row_count           10.0         1.647765e+09   2022-03-20T16:38:00+08:00     155.000000       3.000000            1.097345
17             cartservice-1          trace.from_frontend-1.duration_p95           10.0         1.647765e+09   2022-03-20T16:30:00+08:00       0.001000       0.000000            0.000000
18   productcatalogservice-1             trace.from_frontend-1.row_count           10.0         1.647765e+09   2022-03-20T16:38:00+08:00     155.000000       3.000000            1.099291
19         checkoutservice-1         trace.from_frontend-0.duration_mean            9.0         1.647765e+09   2022-03-20T16:30:00+08:00       0.106758       0.033860            5.860736, 479, ['adservice-1', 'adservice-2', 'adservice2-0', 'cartservice-0', 'cartservice-1', 'cartservice-2', 'cartservice2-0', 'checkoutservice-0', 'checkoutservice-1', 'checkoutservice-2', 'checkoutservice2-0', 'currencyservice-0', 'currencyservice-1', 'currencyservice-2', 'currencyservice2-0', 'emailservice2-0', 'frontend-0', 'frontend-1', 'frontend-2', 'frontend2-0'])```
```

In [ ]:
# In[4]:

import pandas as pd
import numpy as np
import pytz

# reuse existing variables if present
try:
    df_log
except NameError:
    df_log = pd.read_csv("log.csv")
try:
    df_error
except NameError:
    df_error = pd.read_csv("error_logs.csv")
try:
    tz
except NameError:
    tz = pytz.timezone('Asia/Shanghai')

# ensure numeric
df_log['value'] = pd.to_numeric(df_log['value'], errors='coerce')
df_log = df_log.dropna(subset=['value'])

# incident window in Asia/Shanghai -> convert to UTC epoch seconds
start_local = tz.localize(pd.to_datetime("2022-03-20 16:30:00"))
end_local = tz.localize(pd.to_datetime("2022-03-20 17:00:00"))
start_epoch = int(start_local.astimezone(pytz.UTC).timestamp())
end_epoch = int(end_local.astimezone(pytz.UTC).timestamp())

# 1) Filter log.csv for window and log_name == 'log.error_count'
df_log_window = df_log[
    (df_log['timestamp'] >= start_epoch) &
    (df_log['timestamp'] <= end_epoch) &
    (df_log['log_name'] == 'log.error_count')
].copy()

# aggregate per cmdb_id
agg_sum = df_log_window.groupby('cmdb_id').agg(
    error_count_sum=('value','sum'),
    number_of_minutes_reported=('value','count'),
    max_value=('value','max')
).reset_index()

# earliest epoch where value>0 per cmdb_id
positive = df_log_window[df_log_window['value']>0].copy()
earliest_pos = positive.groupby('cmdb_id')['timestamp'].min().reset_index().rename(columns={'timestamp':'earliest_epoch_with_error_count_gt0'})

# merge
agg_log = agg_sum.merge(earliest_pos, on='cmdb_id', how='left')

# add asia/shanghai string for earliest epoch
agg_log['earliest_asia_shanghai'] = agg_log['earliest_epoch_with_error_count_gt0'].apply(
    lambda t: pd.to_datetime(int(t), unit='s', utc=True).tz_convert(tz).isoformat() if not pd.isna(t) else None
)

# keep only rows where error_count_sum > 0, sort by earliest_epoch asc, limit top 20
agg_log_pos = agg_log[agg_log['error_count_sum'] > 0].copy()
agg_log_pos_sorted = agg_log_pos.sort_values(by='earliest_epoch_with_error_count_gt0', ascending=True).head(20).reset_index(drop=True)

# 2) total number of distinct cmdb_id with error_count_sum > 0 and list (limit 20)
total_cmdb_with_errors = agg_log_pos.shape[0]
cmdbs_with_errors_list = sorted(agg_log_pos['cmdb_id'].unique())[:20]

# 3) error_logs.csv: unique (cmdb_id, message) samples with earliest timestamp for each (cmdb_id, message)
if not df_error.empty:
    # ensure message column exists
    df_error['message'] = df_error['message'].astype(str)
    df_error['timestamp'] = pd.to_numeric(df_error['timestamp'], errors='coerce')
    df_error = df_error.dropna(subset=['timestamp'])
    err_grp = df_error.groupby(['cmdb_id','message'], as_index=False).agg(earliest_epoch=('timestamp','min'))
    # add asia shanghai string
    err_grp['earliest_asia_shanghai'] = err_grp['earliest_epoch'].apply(
        lambda t: pd.to_datetime(int(t), unit='s', utc=True).tz_convert(tz).isoformat()
    )
    # sort by earliest timestamp and limit to 20 samples
    error_logs_samples = err_grp.sort_values('earliest_epoch').head(20).reset_index(drop=True)
    # overall earliest and latest timestamps in error_logs.csv
    overall_min_epoch = int(df_error['timestamp'].min())
    overall_max_epoch = int(df_error['timestamp'].max())
    overall_min_asia = pd.to_datetime(overall_min_epoch, unit='s', utc=True).tz_convert(tz).isoformat()
    overall_max_asia = pd.to_datetime(overall_max_epoch, unit='s', utc=True).tz_convert(tz).isoformat()
    error_logs_minmax = pd.DataFrame([{
        'earliest_epoch': overall_min_epoch,
        'earliest_asia_shanghai': overall_min_asia,
        'latest_epoch': overall_max_epoch,
        'latest_asia_shanghai': overall_max_asia
    }])
else:
    error_logs_samples = pd.DataFrame(columns=['cmdb_id','message','earliest_epoch','earliest_asia_shanghai'])
    error_logs_minmax = pd.DataFrame([])

# compact outputs
agg_log_pos_sorted_selected = agg_log_pos_sorted[[
    'cmdb_id','error_count_sum','number_of_minutes_reported',
    'earliest_epoch_with_error_count_gt0','earliest_asia_shanghai','max_value'
]]

agg_log_pos_sorted_selected, total_cmdb_with_errors, cmdbs_with_errors_list, error_logs_samples, error_logs_minmax

```
Out[4]:
```
- Log metrics (window 2022-03-20 16:30–17:00 Asia/Shanghai) — aggregated for log.error_count (only rows with sum>0, sorted by earliest):
  1. frontend-0 — error_count_sum: 26.0; minutes_reported: 31; earliest error-minute ≈ 2022-03-20T16:32:00+08:00; max per-minute value: 7.0
  2. frontend-1 — error_count_sum: 10.0; minutes_reported: 31; earliest error-minute ≈ 2022-03-20T16:32:00+08:00; max per-minute value: 5.0
  3. frontend-2 — error_count_sum: 24.0; minutes_reported: 30; earliest error-minute ≈ 2022-03-20T16:34:00+08:00; max per-minute value: 15.0

- Summary counts:
  - Total distinct cmdb_id with error_count_sum > 0 in the window: 3
  - Those cmdb_id (first 20): ['frontend-0', 'frontend-1', 'frontend-2']

- error_logs.csv (unique (cmdb_id, message) samples with earliest timestamp per pair, up to 20):
  - frontend-0 — "severity: error, message: request error" — earliest 1647765170 → 2022-03-20T16:32:50+08:00
  - frontend-1 — "severity: error, message: request error" — earliest 1647765170 → 2022-03-20T16:32:50+08:00
  - frontend-2 — "severity: error, message: request error" — earliest 1647765241 → 2022-03-20T16:34:01+08:00

- error_logs overall range:
  - earliest error_logs timestamp: 1647765170 → 2022-03-20T16:32:50+08:00
  - latest error_logs timestamp:   1647765433 → 2022-03-20T16:37:13+08:00

Conclusion: frontend-0/1/2 show both metric-level error count increases and matching error log messages during ~16:32–16:34+08:00, indicating frontend request errors concentrated in that brief interval.

The original code execution output of IPython Kernel is also provided below for reference:

(      cmdb_id  error_count_sum  number_of_minutes_reported  earliest_epoch_with_error_count_gt0     earliest_asia_shanghai  max_value
0  frontend-0             26.0                          31                         1.647765e+09  2022-03-20T16:32:00+08:00        7.0
1  frontend-1             10.0                          31                         1.647765e+09  2022-03-20T16:32:00+08:00        5.0
2  frontend-2             24.0                          30                         1.647765e+09  2022-03-20T16:34:00+08:00       15.0, 3, ['frontend-0', 'frontend-1', 'frontend-2'],       cmdb_id                                  message  earliest_epoch     earliest_asia_shanghai
0  frontend-0  severity: error, message: request error      1647765170  2022-03-20T16:32:50+08:00
1  frontend-1  severity: error, message: request error      1647765170  2022-03-20T16:32:50+08:00
2  frontend-2  severity: error, message: request error      1647765241  2022-03-20T16:34:01+08:00,    earliest_epoch     earliest_asia_shanghai  latest_epoch       latest_asia_shanghai
0      1647765170  2022-03-20T16:32:50+08:00    1647765433  2022-03-20T16:37:13+08:00)```
```

In [ ]:
# In[5]:

import pandas as pd
import numpy as np
import pytz
import re

# reuse df_metric and tz if available, otherwise load/define
try:
    df_metric
except NameError:
    df_metric = pd.read_csv("metric.csv")
try:
    tz
except NameError:
    tz = pytz.timezone('Asia/Shanghai')

# ensure numeric values
df_metric['value'] = pd.to_numeric(df_metric['value'], errors='coerce')
df_metric = df_metric.dropna(subset=['value'])

# target cmdb_ids and keyword list (case-insensitive)
cmdb_targets = ['frontend-0','frontend-1','frontend-2','frontend2-0','checkoutservice-1','node-1','node-5']
keywords = ['cpu','mem','memory','net','packet','retransmit','drop','loss','rx','tx','io','disk']
# build regex for matching any keyword as substring (case-insensitive)
kw_regex = '(' + '|'.join(re.escape(k) for k in keywords) + ')'

# 1) find matching kpi_name values per cmdb_id
mask = df_metric['cmdb_id'].isin(cmdb_targets) & df_metric['kpi_name'].str.contains(kw_regex, case=False, na=False)
matched = df_metric[mask].copy()
pairs_df = matched[['cmdb_id','kpi_name']].drop_duplicates().sort_values(['cmdb_id','kpi_name']).reset_index(drop=True)

# compact per-cmdb summary of matched kpis (limit samples to first 20)
matched_kpis_df = pairs_df.groupby('cmdb_id').agg(
    num_kpis=('kpi_name','nunique'),
    kpi_samples=('kpi_name', lambda s: sorted(s.tolist())[:20])
).reset_index()

# 2) compute global P05 and P95 for all (cmdb_id,kpi_name) then filter to our matched pairs
thresholds_all = df_metric.groupby(['cmdb_id','kpi_name'])['value'].agg(
    p05=lambda x: x.quantile(0.05),
    p95=lambda x: x.quantile(0.95)
).reset_index()

# keep only thresholds for matched pairs
thresholds_selected = thresholds_all.merge(pairs_df, on=['cmdb_id','kpi_name'], how='inner')

# 3) filter metric points to incident window (Asia/Shanghai 2022-03-20 16:30 - 17:00) and detect anomalies
start_local = tz.localize(pd.to_datetime("2022-03-20 16:30:00"))
end_local = tz.localize(pd.to_datetime("2022-03-20 17:00:00"))
start_epoch = int(start_local.astimezone(pytz.UTC).timestamp())
end_epoch = int(end_local.astimezone(pytz.UTC).timestamp())

df_window = df_metric[(df_metric['timestamp'] >= start_epoch) & (df_metric['timestamp'] <= end_epoch)].copy()

# join windowed data with thresholds_selected to keep only matching pairs
d = df_window.merge(thresholds_selected, on=['cmdb_id','kpi_name'], how='inner')

# detect anomalies and compute ratio over p95 where applicable
d['is_anom'] = (d['value'] > d['p95']) | (d['value'] < d['p05'])
d['ratio_over_p95'] = np.where((d['p95'] > 0) & (d['value'] > d['p95']), d['value'] / d['p95'], np.nan)

# 4) aggregate per (cmdb_id,kpi_name)
def agg_group(g):
    anom_count = int(g['is_anom'].sum())
    if anom_count > 0:
        earliest_epoch = int(g.loc[g['is_anom'],'timestamp'].min())
        earliest_asia = pd.to_datetime(earliest_epoch, unit='s', utc=True).tz_convert(tz).isoformat()
    else:
        earliest_epoch = np.nan
        earliest_asia = None
    return pd.Series({
        'anomaly_count': anom_count,
        'earliest_anom_epoch': earliest_epoch,
        'earliest_anom_asia_shanghai': earliest_asia,
        'max_in_window': float(g['value'].max()),
        'min_in_window': float(g['value'].min()),
        'max_ratio_over_p95': float(g['ratio_over_p95'].max(skipna=True)) if g['ratio_over_p95'].notna().any() else 0.0
    })

agg = d.groupby(['cmdb_id','kpi_name']).apply(agg_group).reset_index()

# 5) return top 50 rows sorted by anomaly_count desc then earliest_anom_epoch asc
agg_sorted = agg.sort_values(by=['anomaly_count','earliest_anom_epoch'], ascending=[False, True]).reset_index(drop=True)
agg_top50 = agg_sorted.head(50)

# totals and affected cmdb list
total_scanned = thresholds_selected.shape[0]  # distinct (cmdb_id,kpi_name) scanned
anomalous_cmdbs = sorted(agg[agg['anomaly_count']>0]['cmdb_id'].unique())[:20]

# compact selection of columns for display
agg_top50_selected = agg_top50[['cmdb_id','kpi_name','anomaly_count','earliest_anom_epoch','earliest_anom_asia_shanghai','max_in_window','min_in_window','max_ratio_over_p95']]

# Outputs (compact): matched kpis per cmdb, top aggregated anomalies (top50), total scanned count, anomalous cmdb list
matched_kpis_df, agg_top50_selected, total_scanned, anomalous_cmdbs

```
Out[5]:
```
Summary of analysis for the requested cmdb_ids using metric.csv (keywords: cpu, mem, memory, net, packet, retransmit, drop, loss, rx, tx, io, disk):

- Matched KPI counts (unique kpi_name that contain any keyword):
  - checkoutservice-1: 113 matched KPIs (examples: container.node-6.container_cpu_cfs_periods, ...)
  - frontend-0: 161 matched KPIs
  - frontend-1: 161 matched KPIs
  - frontend-2: 161 matched KPIs
  - frontend2-0: 169 matched KPIs
  - node-1: 39 matched KPIs (examples: system.cpu.iowait, system.cpu.pct_usage, system.net.*)
  - node-5: 39 matched KPIs

- Global thresholds:
  - Global P05/P95 were computed per (cmdb_id, kpi_name) using the full metric series before window filtering.

- Incident-window anomaly scan (window 2022-03-20 16:30:00 → 17:00:00 Asia/Shanghai):
  - Total distinct (cmdb_id, kpi_name) scanned (matching keywords): 843.
  - All seven provided cmdb_id candidates had at least one anomaly in the window: ['checkoutservice-1','frontend-0','frontend-1','frontend-2','frontend2-0','node-1','node-5'] (limited to first 20 — here all are shown).
  - Total results returned (top examples limited to 50, sorted by anomaly_count): many KPI pairs show 5–6 anomaly points in the window. Examples (compact):
    1. node-1 | system.cpu.pct_usage — anomaly_count 6; earliest 2022-03-20T16:30:00+08:00; max 36.07; min 11.98; max ratio_over_P95 ≈ 1.20
    2. checkoutservice-1 | mesh.destination.frontend.checkoutservice.istio... — anomaly_count 6; earliest 2022-03-20T16:31:00+08:00; max 59.5; min 0.0; max ratio_over_P95 ≈ 205.0 (very large spike)
    3. node-5 | system.net.packets_in.count — anomaly_count 6; earliest 2022-03-20T16:31:00+08:00; max 196.12; min 97.14; ratio ≈ 1.02
    4. node-5 | system.net.packets_out.count — anomaly_count 6; earliest 2022-03-20T16:31:00+08:00; max 195.58; min 93.09; ratio ≈ 1.02
    5. frontend-1 | container.node-5.container_cpu_cfs_periods — anomaly_count 6; earliest 2022-03-20T16:32:00+08:00; max 224.5; min 17.0; ratio ≈ 1.13
    6. frontend2-0 | mesh.destination.unknown.frontend.istio_request... — anomaly_count 6 (one entry); earliest 2022-03-20T16:32:00+08:00; max ≈ 354.01; max ratio_over_P95 ≈ 10.30 (large)
    7. frontend2-0 | several mesh.source.frontend2.shippingservice2... KPIs — anomaly_count 5; earliest 2022-03-20T16:30:00+08:00; very large absolute values for some request/count KPIs (e.g., 100,750 and 10,075 shown) with ratio ≈ 1.083.

- Observations / actionable highlights:
  - Many CPU- and network-related KPIs for the listed cmdb_ids show anomalies clustered around ~16:30–16:33+08:00.
  - Some anomalies are modest (ratio ~1.02–1.2) but consistent across multiple KPIs and nodes; these suggest broad but moderate deviations (e.g., node CPU, node net packets).
  - A few KPI pairs show very large ratio_over_P95 spikes (notably checkoutservice-1 mesh destination KPI ratio ≈ 205, frontend2-0 and frontend-2 some mesh/request KPIs with ratios >5–10). Those large ratios indicate strong spikes and should be prioritized for investigation.
  - Total scanned set is large (843 KPI pairs), so anomalies are widespread across CPU, memory, network and mesh/request KPIs for these components.

If you want, I can: (a) show the full top-50 table rows (more detail per KPI), (b) produce a concise list of the highest-ratio anomalies only, or (c) focus on a single cmdb_id for deeper inspection.

The original code execution output of IPython Kernel is also provided below for reference:

(             cmdb_id  num_kpis                                        kpi_samples
0  checkoutservice-1       113  [container.node-6.container_cpu_cfs_periods, c...
1         frontend-0       161  [container.node-6.container_cpu_cfs_periods, c...
2         frontend-1       161  [container.node-5.container_cpu_cfs_periods, c...
3         frontend-2       161  [container.node-5.container_cpu_cfs_periods, c...
4        frontend2-0       169  [container.node-6.container_cpu_cfs_periods, c...
5             node-1        39  [system.cpu.iowait, system.cpu.pct_usage, syst...
6             node-5        39  [system.cpu.iowait, system.cpu.pct_usage, syst...,               cmdb_id                                           kpi_name  anomaly_count  earliest_anom_epoch earliest_anom_asia_shanghai  max_in_window  min_in_window  max_ratio_over_p95
0              node-1                               system.cpu.pct_usage            6.0         1.647765e+09   2022-03-20T16:30:00+08:00       36.07000       11.98000            1.199534
1   checkoutservice-1  mesh.destination.frontend.checkoutservice.isti...            6.0         1.647765e+09   2022-03-20T16:31:00+08:00       59.50000        0.00000          204.995693
2              node-5                        system.net.packets_in.count            6.0         1.647765e+09   2022-03-20T16:31:00+08:00      196.12000       97.14000            1.020236
3              node-5                       system.net.packets_out.count            6.0         1.647765e+09   2022-03-20T16:31:00+08:00      195.58000       93.09000            1.024730
4          frontend-1         container.node-5.container_cpu_cfs_periods            6.0         1.647765e+09   2022-03-20T16:32:00+08:00      224.50000       17.00000            1.125313
..                ...                                                ...            ...                  ...                         ...            ...            ...                 ...
45         frontend-1  mesh.source.frontend.recommendationservice.ist...            6.0         1.647765e+09   2022-03-20T16:33:00+08:00       73.00000        2.50000            1.050360
46        frontend2-0  mesh.destination.unknown.frontend2.istio_reque...            5.0         1.647765e+09   2022-03-20T16:30:00+08:00        1.64350        0.03600            5.749519
47        frontend2-0  mesh.source.frontend2.shippingservice2.istio_r...            5.0         1.647765e+09   2022-03-20T16:30:00+08:00   100750.00000    13950.00000            1.083333
48        frontend2-0  mesh.source.frontend2.shippingservice2.istio_r...            5.0         1.647765e+09   2022-03-20T16:30:00+08:00        0.06825        0.00945            1.083333
49        frontend2-0  mesh.source.frontend2.shippingservice2.istio_r...            5.0         1.647765e+09   2022-03-20T16:30:00+08:00    10075.00000     1395.00000            1.083333

[50 rows x 8 columns], 843, ['checkoutservice-1', 'frontend-0', 'frontend-1', 'frontend-2', 'frontend2-0', 'node-1', 'node-5'])

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[6]:

import pandas as pd
import numpy as np
import pytz

# reuse df_metric and tz if available
try:
    df_metric
except NameError:
    df_metric = pd.read_csv("metric.csv")
try:
    tz
except NameError:
    tz = pytz.timezone('Asia/Shanghai')

# ensure numeric and drop NaNs
df_metric['value'] = pd.to_numeric(df_metric['value'], errors='coerce')
df_metric = df_metric.dropna(subset=['value'])

# 1) Global P95 per (cmdb_id, kpi_name) using full series
p95_df = df_metric.groupby(['cmdb_id','kpi_name'], as_index=False)['value'].agg(p95=lambda x: x.quantile(0.95))

# 2) Filter to incident window (Asia/Shanghai 2022-03-20 16:30:00 - 17:00:00) -> convert to UTC epoch
start_local = tz.localize(pd.to_datetime("2022-03-20 16:30:00"))
end_local = tz.localize(pd.to_datetime("2022-03-20 17:00:00"))
start_epoch = int(start_local.astimezone(pytz.UTC).timestamp())
end_epoch = int(end_local.astimezone(pytz.UTC).timestamp())

df_window = df_metric[(df_metric['timestamp'] >= start_epoch) & (df_metric['timestamp'] <= end_epoch)].copy()

# merge p95 into windowed points
w = df_window.merge(p95_df, on=['cmdb_id','kpi_name'], how='left')

# 2/3) identify points where value > P95
w_pos = w[w['value'] > w['p95']].copy()

# if no anomalies, return empty concise outputs
if w_pos.empty:
    top20_df = pd.DataFrame(columns=['cmdb_id','kpi_name','earliest_anom_epoch','earliest_anom_asia_shanghai','p95','max_in_window','max_ratio_over_p95'])
    earliest_overall_epoch = None
    earliest_overall_asia = None
else:
    # 2/3) aggregate per (cmdb_id,kpi_name): earliest anomaly epoch and max in window
    agg = w_pos.groupby(['cmdb_id','kpi_name'], as_index=False).agg(
        earliest_anom_epoch=('timestamp','min'),
        max_in_window=('value','max'),
        p95=('p95','first')  # p95 is constant per group
    )
    # compute ratio, handle p95==0 by setting ratio to NaN
    agg['max_ratio_over_p95'] = np.where(agg['p95']>0, agg['max_in_window'] / agg['p95'], np.nan)
    # asia shanghai string
    agg['earliest_anom_asia_shanghai'] = agg['earliest_anom_epoch'].apply(
        lambda t: pd.to_datetime(int(t), unit='s', utc=True).tz_convert(tz).isoformat()
    )
    # 3) select columns and sort by ratio desc
    top20_df = agg[['cmdb_id','kpi_name','earliest_anom_epoch','earliest_anom_asia_shanghai','p95','max_in_window','max_ratio_over_p95']] \
                .sort_values(by='max_ratio_over_p95', ascending=False).head(20).reset_index(drop=True)
    # compute single earliest anomaly epoch among these top20 rows
    earliest_overall_epoch = int(top20_df['earliest_anom_epoch'].min())
    earliest_overall_asia = pd.to_datetime(earliest_overall_epoch, unit='s', utc=True).tz_convert(tz).isoformat()

# Compact outputs
top20_df, earliest_overall_epoch, earliest_overall_asia

```
Out[6]:
```
Summary of metric-based spikes (value > global P95) during 2022-03-20 16:30–17:00 Asia/Shanghai — top 20 by max_in_window / P95:

- Method: computed global P95 per (cmdb_id,kpi_name) from the full metric.csv, then within the incident window selected points > P95 and for each pair reported earliest anomaly time in-window, P95, max_in_window and max_ratio_over_p95.

- Top results (highest ratios, examples from the top 20):
  1. frontend-1 | container.node-5.container_cpu_cfs_throttled_seconds_total  
     earliest 2022-03-20T16:41:00+08:00; P95 ≈ 9.09e-13; max_in_window = 0.001678; ratio ≈ 1.85e+09 (very large due to near-zero P95).
  2. checkoutservice-1 | mesh.source.checkoutservice.currencyservice.istio_request_count  
     earliest 2022-03-20T16:34:00+08:00; P95 ≈ 0.021675; max_in_window = 59.715; ratio ≈ 2.76e+03.
  3. cartservice2-0 | container.node-5.container_fs_reads./dev/vda  
     earliest 2022-03-20T16:30:00+08:00; P95 = 2.0; max_in_window = 679.0; ratio ≈ 339.5.
  4. cartservice2-0 | container.node-5.container_fs_reads_MB./dev/vda  
     earliest 2022-03-20T16:30:00+08:00; P95 ≈ 0.12695; max_in_window ≈ 41.664; ratio ≈ 328.2.
  5. node-5 | system.io.rkb_s  
     earliest 2022-03-20T16:40:00+08:00; P95 = 48.0; max_in_window = 15088.0; ratio ≈ 314.33.
  6. node-5 | system.io.r_s  
     earliest 2022-03-20T16:40:00+08:00; P95 = 1.5; max_in_window = 362.5; ratio ≈ 241.67.
  7. frontend-1 | mesh.source.frontend.currencyservice.istio_request_count  
     earliest 2022-03-20T16:32:00+08:00; P95 ≈ 0.385525; max_in_window ≈ 91.795; ratio ≈ 238.10.
  8. checkoutservice-1 | mesh.destination.frontend.checkoutservice.istio_request_count  
     earliest 2022-03-20T16:31:00+08:00; P95 ≈ 0.29025; max_in_window = 59.5; ratio ≈ 204.996.
  9. node-1 | system.net.tcp.retrans_segs  
     earliest 2022-03-20T16:31:00+08:00; P95 = 0.13; max_in_window ≈ 25.73; ratio ≈ 197.92.
  10. frontend-2 | mesh.source.frontend.checkoutservice.istio_request_count  
     earliest 2022-03-20T16:34:00+08:00; P95 ≈ 0.4035; max_in_window = 59.5; ratio ≈ 147.46.
  (other top-20 entries include high ratios for various io, network receive, mesh request/count KPIs and node IO/util metrics — ratios range down to ~21.)

- Earliest anomaly among these top 20: epoch 1647765000 → 2022-03-20T16:30:00+08:00.

- Interpretation / highlights:
  - Many extreme ratios are driven by very small P95 baselines (near-zero) for some counters, producing huge ratios; treat those carefully (they reflect large absolute spikes but can be magnified by tiny denominators).
  - High absolute spikes appear on IO (fs reads, rkb_s), network retransmission, and mesh request/count KPIs for frontends and checkoutservice — these warrant priority investigation.
  - Node-level and container-level I/O and network metrics (node-5, node-1, cartservice2-0, etc.) show especially large deviations in this window.

If you want, I can return the full top-20 table rows (detailed values) or isolate the pairs with very small P95 to inspect absolute vs. relative significance.

The original code execution output of IPython Kernel is also provided below for reference:

(              cmdb_id                                           kpi_name  earliest_anom_epoch earliest_anom_asia_shanghai           p95  max_in_window  max_ratio_over_p95
0          frontend-1  container.node-5.container_cpu_cfs_throttled_s...           1647765660   2022-03-20T16:41:00+08:00  9.094947e-13       0.001678        1.845257e+09
1   checkoutservice-1  mesh.source.checkoutservice.currencyservice.is...           1647765240   2022-03-20T16:34:00+08:00  2.167500e-02      59.715000        2.755017e+03
2      cartservice2-0       container.node-5.container_fs_reads./dev/vda           1647765000   2022-03-20T16:30:00+08:00  2.000000e+00     679.000000        3.395000e+02
3      cartservice2-0    container.node-5.container_fs_reads_MB./dev/vda           1647765000   2022-03-20T16:30:00+08:00  1.269531e-01      41.664062        3.281846e+02
4              node-5                                    system.io.rkb_s           1647765600   2022-03-20T16:40:00+08:00  4.800000e+01   15088.000000        3.143333e+02
5              node-5                                      system.io.r_s           1647765600   2022-03-20T16:40:00+08:00  1.500000e+00     362.500000        2.416667e+02
6          frontend-1  mesh.source.frontend.currencyservice.istio_req...           1647765120   2022-03-20T16:32:00+08:00  3.855250e-01      91.794975        2.381038e+02
7   checkoutservice-1  mesh.destination.frontend.checkoutservice.isti...           1647765060   2022-03-20T16:31:00+08:00  2.902500e-01      59.500000        2.049957e+02
8              node-1                        system.net.tcp.retrans_segs           1647765060   2022-03-20T16:31:00+08:00  1.300000e-01      25.730000        1.979231e+02
9          frontend-2  mesh.source.frontend.checkoutservice.istio_req...           1647765240   2022-03-20T16:34:00+08:00  4.035000e-01      59.500000        1.474597e+02
10         frontend-0  mesh.source.frontend.currencyservice.istio_req...           1647765120   2022-03-20T16:32:00+08:00  6.111500e-01      59.534550        9.741397e+01
11    checkoutservice                                       app.grpc.mrt           1647765120   2022-03-20T16:32:00+08:00  9.900000e-02       7.469563        7.545013e+01
12  checkoutservice-0  mesh.source.checkoutservice.currencyservice.is...           1647765120   2022-03-20T16:32:00+08:00  1.930000e-02       0.833325        4.317746e+01
13  checkoutservice-0  mesh.source.checkoutservice.jaeger-collector.i...           1647766200   2022-03-20T16:50:00+08:00  1.575000e-03       0.061500        3.904762e+01
14             node-3                        system.net.tcp.retrans_segs           1647765180   2022-03-20T16:33:00+08:00  2.000000e-01       6.970000        3.485000e+01
15             node-5                                     system.io.util           1647765600   2022-03-20T16:40:00+08:00  1.050000e+00      28.350000        2.700000e+01
16   paymentservice-0  container.node-6.container_network_receive_MB....           1647765060   2022-03-20T16:31:00+08:00  2.709055e-02       0.598999        2.211100e+01
17   paymentservice-1  container.node-6.container_network_receive_MB....           1647765060   2022-03-20T16:31:00+08:00  2.787161e-02       0.611630        2.194457e+01
18         frontend-1  mesh.destination.unknown.frontend.istio_reques...           1647765360   2022-03-20T16:36:00+08:00  4.566000e+00      95.991750        2.102316e+01
19  paymentservice2-0  container.node-6.container_network_receive_MB....           1647765360   2022-03-20T16:36:00+08:00  3.007698e-02       0.632100        2.101608e+01, 1647765000, '2022-03-20T16:30:00+08:00')```
```

In [ ]:
# In[7]:

import pandas as pd
import numpy as np
import pytz

# reuse df_metric and tz if available
try:
    df_metric
except NameError:
    df_metric = pd.read_csv("metric.csv")
try:
    tz
except NameError:
    tz = pytz.timezone('Asia/Shanghai')

# ensure numeric values
df_metric['value'] = pd.to_numeric(df_metric['value'], errors='coerce')
df_metric = df_metric.dropna(subset=['value'])

# 1) filter kpi_name containing retrans keywords and node-like cmdb_id
kw_regex = r'(retrans|retrans_segs|tcp\.retrans)'
mask_full = df_metric['kpi_name'].str.contains(kw_regex, case=False, na=False) & df_metric['cmdb_id'].str.startswith('node-')
matched_full = df_metric[mask_full].copy()

# If none matched, return empty results
if matched_full.empty:
    node_summary_selected = pd.DataFrame(columns=['cmdb_id','anomaly_count','earliest_anom_epoch','earliest_anom_asia_shanghai','p95','max_in_window','max_ratio_over_p95'])
    earliest_overall_epoch = None
    earliest_overall_asia = None
else:
    # 1) compute global P95 per (cmdb_id,kpi_name) using full series (before window filter)
    p95_df = matched_full.groupby(['cmdb_id','kpi_name'], as_index=False)['value'].agg(p95=lambda x: x.quantile(0.95))

    # 2) filter to incident window (Asia/Shanghai -> convert to UTC epoch)
    start_local = tz.localize(pd.to_datetime("2022-03-20 16:30:00"))
    end_local = tz.localize(pd.to_datetime("2022-03-20 17:00:00"))
    start_epoch = int(start_local.astimezone(pytz.UTC).timestamp())
    end_epoch = int(end_local.astimezone(pytz.UTC).timestamp())

    df_window = df_metric[(df_metric['timestamp'] >= start_epoch) & (df_metric['timestamp'] <= end_epoch)].copy()

    # keep only node retrans KPIs in window by merging with p95_df
    window_matched = df_window.merge(p95_df, on=['cmdb_id','kpi_name'], how='inner')

    # 3) detect points where value > P95
    window_anom = window_matched[window_matched['value'] > window_matched['p95']].copy()

    if window_anom.empty:
        node_summary_selected = pd.DataFrame(columns=['cmdb_id','anomaly_count','earliest_anom_epoch','earliest_anom_asia_shanghai','p95','max_in_window','max_ratio_over_p95'])
        earliest_overall_epoch = None
        earliest_overall_asia = None
    else:
        # per (node,kpi) aggregates
        per_pair = window_anom.groupby(['cmdb_id','kpi_name'], as_index=False).agg(
            anomaly_count=('value','count'),
            earliest_anom_epoch=('timestamp','min'),
            max_in_window=('value','max'),
            p95=('p95','first')
        )
        per_pair['max_ratio_over_p95'] = np.where(per_pair['p95']>0, per_pair['max_in_window']/per_pair['p95'], np.nan)

        # per-node totals and earliest
        node_totals = per_pair.groupby('cmdb_id', as_index=False).agg(
            anomaly_count_total=('anomaly_count','sum'),
            earliest_anom_epoch_node=('earliest_anom_epoch','min')
        )

        # select for each node the KPI with highest max_ratio_over_p95
        per_pair_sorted = per_pair.sort_values(['cmdb_id','max_ratio_over_p95'], ascending=[True,False])
        top_kpi_per_node = per_pair_sorted.groupby('cmdb_id', as_index=False).first()[['cmdb_id','p95','max_in_window','max_ratio_over_p95','kpi_name']]

        # merge node totals with top_kpi info
        node_summary = node_totals.merge(top_kpi_per_node, on='cmdb_id', how='left')

        # add asia shanghai string for earliest_anom_epoch_node
        node_summary['earliest_anom_asia_shanghai'] = node_summary['earliest_anom_epoch_node'].apply(
            lambda t: pd.to_datetime(int(t), unit='s', utc=True).tz_convert(tz).isoformat()
        )

        # prepare final columns as requested
        node_summary_selected = node_summary.rename(columns={
            'anomaly_count_total':'anomaly_count',
            'earliest_anom_epoch_node':'earliest_anom_epoch',
        })[[
            'cmdb_id','anomaly_count','earliest_anom_epoch','earliest_anom_asia_shanghai','p95','max_in_window','max_ratio_over_p95'
        ]]

        # sort by max_ratio_over_p95 desc and limit to top 10
        node_summary_selected = node_summary_selected.sort_values('max_ratio_over_p95', ascending=False).head(10).reset_index(drop=True)

        # single earliest anomaly epoch among all node retrans anomalies
        earliest_overall_epoch = int(window_anom['timestamp'].min())
        earliest_overall_asia = pd.to_datetime(earliest_overall_epoch, unit='s', utc=True).tz_convert(tz).isoformat()

# compact outputs
node_summary_selected, earliest_overall_epoch, earliest_overall_asia

```
Out[7]:
```
Summary (node-level network retransmission KPIs; global P95 from full series; window 2022-03-20 16:30–17:00 Asia/Shanghai):

Top nodes by max_ratio_over_p95 (only nodes with value > global P95 in-window):

1. node-1 — anomaly_count: 2; earliest: 1647765060 → 2022-03-20T16:31:00+08:00; P95: 0.13; max_in_window: 25.73; max_ratio_over_p95 ≈ 197.92  
2. node-3 — anomaly_count: 2; earliest: 1647765180 → 2022-03-20T16:33:00+08:00; P95: 0.20; max_in_window: 6.97; max_ratio_over_p95 = 34.85  
3. node-4 — anomaly_count: 1; earliest: 1647765060 → 2022-03-20T16:31:00+08:00; P95: 0.02; max_in_window: 0.20; max_ratio_over_p95 = 10.00  
4. node-2 — anomaly_count: 2; earliest: 1647765060 → 2022-03-20T16:31:00+08:00; P95: 0.13; max_in_window: 0.69; max_ratio_over_p95 ≈ 5.31  
5. node-6 — anomaly_count: 3; earliest: 1647765180 → 2022-03-20T16:33:00+08:00; P95: 0.57; max_in_window: 0.88; max_ratio_over_p95 ≈ 1.54

Earliest retransmission anomaly among all node results: 1647765060 → 2022-03-20T16:31:00+08:00.

Interpretation: node-1 exhibits the largest retransmission spike by ratio (very large absolute retrans_segs at ~16:31+08:00). node-3 and node-4 also show notable spikes; other nodes show smaller but above-P95 retransmission increases. These node-level retransmission events around ~16:31–16:33+08:00 merit network-level investigation (packet retransmission, link congestion, or upstream issues).

The original code execution output of IPython Kernel is also provided below for reference:

(  cmdb_id  anomaly_count  earliest_anom_epoch earliest_anom_asia_shanghai   p95  max_in_window  max_ratio_over_p95
0  node-1              2           1647765060   2022-03-20T16:31:00+08:00  0.13          25.73          197.923077
1  node-3              2           1647765180   2022-03-20T16:33:00+08:00  0.20           6.97           34.850000
2  node-4              1           1647765060   2022-03-20T16:31:00+08:00  0.02           0.20           10.000000
3  node-2              2           1647765060   2022-03-20T16:31:00+08:00  0.13           0.69            5.307692
4  node-6              3           1647765180   2022-03-20T16:33:00+08:00  0.57           0.88            1.543860, 1647765060, '2022-03-20T16:31:00+08:00')```
```